In [ ]:
with open("input.txt") as f:
    text = f.read().replace("\n", " ")

print(text[:100])

In [ ]:
class Tokenizer:
    """Character Level Tokenizer"""

    def __init__(self, text: str):
        self.UNK = "<UNK>"
        self.chars = sorted(list(set(text)))
        self.chars += [self.UNK]
        self.size = len(self.chars)
        self.char_to_id = {c: i for i, c in enumerate(self.chars)}
        self.id_to_char = {i: c for i, c in enumerate(self.chars)}

    def encode(self, text: str) -> list[int]:
        return [self.char_to_id.get(c, self.char_to_id[self.UNK]) for c in text]

    def decode(self, ids: list[int]) -> str:
        return "".join(self.id_to_char.get(i, self.UNK) for i in ids)

In [ ]:
import torch
import torch.nn as nn

tokenizer = Tokenizer(text)
print(tokenizer.size)

In [ ]:
import torch.nn.functional as F


class SelfAttention(nn.Module):
    def __init__(self, embed_size: int):
        super(SelfAttention, self).__init__()
        self.embed_size = embed_size
        # size should not matter
        self.Wq = nn.Linear(embed_size, embed_size)
        self.Wk = nn.Linear(embed_size, embed_size)
        self.Wv = nn.Linear(embed_size, embed_size)

    def forward(self, x):
        # x: (bs, seq_len, embed_size)

        q = self.Wq(x)
        k = self.Wk(x)
        v = self.Wv(x)  # (bs, seq_len, embed_size)

        # unnormalized attention weights
        omega = q @ k.transpose(1, 2)  # (bs, seq_len, seq_len)
        mask = torch.triu(torch.ones(omega.shape[1], omega.shape[1]), diagonal=1).to(
            x.device
        )
        masked = omega.masked_fill(mask.bool(), -torch.inf)
        alfa = F.softmax(
            masked / self.embed_size**0.5, dim=-1
        )  # (bs, seq_len, seq_len)
        z = alfa @ v  # (bs, seq_len, embed_size)
        return z  # (bs, seq_len, embed_size)


class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size: int, nheads: int):
        super(MultiHeadAttention, self).__init__()
        assert embed_size % nheads == 0
        self.nheads = nheads
        self.head_dim = embed_size // nheads
        self.heads = nn.ModuleList(
            [SelfAttention(self.head_dim) for _ in range(nheads)]
        )

    def forward(self, x):
        # (bs, seq_len, embed_dim)
        bs, seq_len, embed_dim = x.shape
        x_transposed = x.view(bs, seq_len, self.nheads, self.head_dim).transpose(
            1, 2
        )  # (bs, nheads, seq_len, head_dim)
        return torch.cat(
            [head(x_transposed[:, i, :, :]) for i, head in enumerate(self.heads)],
            dim=-1,
        )


class MLP(nn.Module):
    def __init__(self, embed_size: int):
        super(MLP, self).__init__()
        self.l1 = nn.Linear(embed_size, 4 * embed_size)
        self.gelu = nn.GELU()
        self.l2 = nn.Linear(4 * embed_size, embed_size)
        # todo: optional dropout

    def forward(self, x):
        x = self.l1(x)
        x = self.gelu(x)
        x = self.l2(x)
        return x


class Block(nn.Module):
    def __init__(self, embed_size: int):
        super(Block, self).__init__()
        self.embed_size = embed_size
        self.norm1 = nn.LayerNorm(self.embed_size)
        self.mha = MultiHeadAttention(self.embed_size, 4)
        self.mlp = MLP(self.embed_size)
        self.norm2 = nn.LayerNorm(self.embed_size)

    def forward(self, x):
        # x: (bs, seq_len, embed_dim)
        x = x + self.mha(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class Transformer(nn.Module):
    def __init__(
        self, embed_size: int, vocab_size: int, max_seq_len: int, nblocks: int
    ):
        super(Transformer, self).__init__()
        self.embedding = nn.Embedding(tokenizer.size, embed_size)
        self.pos_embedding = nn.Embedding(max_seq_len, embed_size)
        self.blocks = nn.ModuleList([Block(embed_size) for _ in range(nblocks)])
        self.norm = nn.LayerNorm(embed_size)
        self.output = nn.Linear(embed_size, vocab_size)

    def forward(self, x):
        # x: (bs, seq_len)
        positions = torch.arange(0, x.size(1), device=x.device).unsqueeze(0)
        emb = self.embedding(x) + self.pos_embedding(positions)
        for block in self.blocks:
            emb = block(emb)
        emb = self.norm(emb)
        emb = self.output(emb)  # bs, seq_len, vocab_size
        return emb

In [ ]:
import gc

gc.collect()
with torch.no_grad():
    torch.cuda.empty_cache()

In [ ]:
from torch.utils.data import DataLoader, Dataset


class TextDataset(Dataset):
    def __init__(self, tokens: list[int], seq_len: int):
        self.tokens = tokens
        self.seq_len = seq_len

    def __len__(self):
        return len(self.tokens) - seq_len

    def __getitem__(self, idx: int):
        chunk = self.tokens[idx : idx + self.seq_len + 1]
        x = torch.tensor(chunk[:-1])
        y = torch.tensor(chunk[1:])
        return x, y


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
tokens = tokenizer.encode(text)
seq_len = 128
train_data = TextDataset(tokens, seq_len)
bs = 64
train_loader = DataLoader(
    train_data, batch_size=bs, shuffle=True, generator=torch.Generator(device=device)
)

In [ ]:
import torch.optim as optim


embedding_dim = 256
nblocks = 4
model = Transformer(embedding_dim, tokenizer.size, seq_len, nblocks).to(device)
lr = 1e-4
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

nepochs = 100
for epoch in range(1, nepochs + 1):
    running_loss = 0.0
    for i, (x, y) in enumerate(train_loader):
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits.view(-1, tokenizer.size), y.view(-1))
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch}, loss: {running_loss / len(train_loader)}")